## Modelo CNN clásico desde cero

In [21]:
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision import models
import torch
import torch.nn as nn
import torch.optim as optim
import sklearn.metrics as metrics
from PIL import Image
from tqdm import tqdm

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", DEVICE)

Usando dispositivo: cuda


## Caracteristicas 

In [3]:
img_size = 224
batch_size = 64
epochs = 3
LR = 1e-3
num_classes = 4

## Transformaciones para aumentacion de datos

In [4]:
transform_train = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

## Cargar datasets

In [5]:
dataset = load_dataset("mirluvams/popocatepetl")
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 216352
    })
    test: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 24040
    })
})

In [6]:
print(dataset)
print(dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 216352
    })
    test: Dataset({
        features: ['label', 'image', 'video', 'id'],
        num_rows: 24040
    })
})
{'label': ClassLabel(names=['vapor', 'ceniza', 'sin_emision', 'no_visible']), 'image': Image(mode=None, decode=True), 'video': Value('string'), 'id': Value('int32')}


In [7]:
def transform_examples(batch, transform):
    images = [transform(img.convert("RGB")) for img in batch["image"]]
    labels = batch["label"]
    return {"pixel_values": images, "label": labels}

In [8]:
train_dataset = dataset["train"].with_transform(lambda x: transform_examples(x, transform_train))
test_dataset = dataset["test"].with_transform(lambda x: transform_examples(x, transform_test))

In [9]:
def collate_fn(batch):
    images = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch])
    return images, labels


In [10]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

## Creacion del modelo

In [11]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(128 * (img_size // 8) * (img_size // 8), 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x


In [12]:
num_classes = len(set(dataset["train"]["label"]))
model = SimpleCNN(num_classes=num_classes).to(DEVICE)
model

SimpleCNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=100352, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=256, out_features=4, bias=True)
  )
)

In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [14]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    print("EPOCH", epoch+1)
    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Época [{epoch+1}/{epochs}] - Pérdida: {running_loss/len(train_loader):.4f}")

EPOCH 1


100%|██████████| 3381/3381 [30:01<00:00,  1.88it/s]


Época [1/3] - Pérdida: 0.2470
EPOCH 2


100%|██████████| 3381/3381 [29:03<00:00,  1.94it/s]


Época [2/3] - Pérdida: 0.1350
EPOCH 3


100%|██████████| 3381/3381 [28:42<00:00,  1.96it/s]

Época [3/3] - Pérdida: 0.1092


In [29]:
model.eval()
preds, labels = [], []

with torch.no_grad():
    for images, batch_labels in tqdm(test_loader):
        images, batch_labels = images.to(DEVICE), batch_labels.to(DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        preds.extend(predicted.cpu().numpy())
        labels.extend(batch_labels.cpu().numpy())

100%|██████████| 376/376 [02:34<00:00,  2.44it/s]


In [31]:
acc = metrics.accuracy_score(labels, preds)
f1 = metrics.f1_score(labels, preds, average="weighted")
f1_micro = metrics.f1_score(labels, preds, average="micro")
f1_macro = metrics.f1_score(labels, preds, average="macro")

print(f"Accuracy: {acc:.4f} | F1: {f1:.4f} | F1_micro: {f1_micro:.4f} | F1_macro: {f1_macro:.4f}")
#{"model": model_name, "accuracy": acc, "f1": f1, "f1_micro": f1_micro, "f1_macro": f1_macro}

Accuracy: 0.9666 | F1: 0.9665 | F1_micro: 0.9666 | F1_macro: 0.9544


In [32]:
torch.save(model.state_dict(), "cnn_popocatepetl.pth")
print("Modelo guardado como cnn_popocatepetl.pth")

Modelo guardado como cnn_popocatepetl.pth
